# 04C — Validation verrouillée du batch 3 et reconstruction spatiale

Ce notebook constitue l'étape de **validation externe sur le batch 3** du protocole `8tracks_v5`.

Principes verrouillés :

- l'autorité de sélection reste **03B** (`selected_models` / `selected_runs`) ;
- 03C fournit uniquement l'éligibilité de domaine et le verrou spatial ;
- 04A fournit uniquement le statut `supported` / `diagnostic_only` des modèles déjà sélectionnés ;
- 04B reste un **audit de couverture TPE** et n'entre jamais dans la population exécutée en 04C ;
- aucun seuil de décision 03B n'est recalibré avec le batch 3 ;
- aucun nouveau `validation_candidate_id`, `calibration_id`, `data_config_id`, `fit_config_id` ou `projection_config_id` n'est créé ;
- les huit tracks restent indépendants : 04C applique des guardrails **par exécution et par scope**, sans comparaison ni sélection entre tracks ;
- le batch 4 n'est jamais chargé ;
- les guardrails stricts de 04C sont un **contrat d'évaluation enfant versionné** dans `8tracks_v5`.

Clés canoniques :

- modèle scientifique : `model_id` ;
- exécution : `(model_id, random_state)` ;
- fit technique : `fit_id` ;
- projection continue : `projection_id` ;
- politique de décision : `(model_id, random_state, decision_scope)`.

## A — Initialisation

In [1]:
from __future__ import annotations

import inspect
import json
import platform
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "src").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError(
        "Launch the notebook from the project root or from notebooks/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 40)

from src import experiment_config as expcfg
from src.io.database_h5 import load_nir_uco_h5
from src.protocol_governance import (
    sha256_dataframe,
    sha256_file,
    verify_frozen_protocol,
)
from src.spectra.band_selection import select_wavelength_range_from_database
from src.utils import load_parquet, save_parquet
from src.workflows.simca_calibration_registry import (
    build_validation_execution_registry,
    validate_internal_calibration_manifest,
)
from src.workflows.simca import (
    run_locked_simca_validation_refit,
    run_locked_simca_validation_refit_checkpointed,
)
from src.workflows.simca_candidates import (
    hash_locked_validation_evaluation_rule,
    hash_locked_validation_plan,
)
from src.workflows.simca_thresholds_calibration import build_pixel_vote_table
from src.workflows.simca_grid_evaluation import (
    build_validation_guardrails,
    evaluate_locked_validation_predictions,
)
from src.workflows.spatial_postprocessing_calibration import (
    build_locked_spatial_validation_outputs,
    verify_spatial_postprocessing_lock,
)

%load_ext autoreload
%autoreload 2

print("Python:", platform.python_version())
print("PROJECT_ROOT:", PROJECT_ROOT)

Python: 3.14.6
PROJECT_ROOT: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts


## B — Contrat 04C, chemins et verrous

Cette cellule vérifie que le code chargé correspond bien à la nouvelle architecture compacte.  
Elle ne modifie pas le protocole global `8tracks_v5` : le hash global reste celui du verrou déjà produit, tandis que le profil de guardrails 04C possède son propre hash d'évaluation.

In [2]:
if str(expcfg.PROTOCOL_VERSION) != "8tracks_v5":
    raise RuntimeError(
        f"This notebook is written for 8tracks_v5, got {expcfg.PROTOCOL_VERSION!r}."
    )

if str(expcfg.SIMCA_CONCAT_REFIT_EXECUTION_POLICY) != (
    "selected_03B_executions_no_04C_reselection"
):
    raise RuntimeError("04C execution policy is not the compact v5 policy.")

if not hasattr(expcfg, "SIMCA_CONCAT_REFIT_GUARDRAIL_PROFILE_ID"):
    raise RuntimeError("Missing the versioned strict 04C guardrail profile.")

amendment = dict(expcfg.SIMCA_CONCAT_REFIT_EVALUATION_AMENDMENT)
if not bool(amendment.get("guardrail_thresholds_changed", False)):
    raise RuntimeError("The strict 04C guardrail amendment is not active.")
if amendment.get("batch3_used_to_choose_thresholds") is not False:
    raise RuntimeError(
        "Prospective validation requires batch3_used_to_choose_thresholds=False."
    )

expected_signatures = {
    "build_validation_execution_registry": (
        build_validation_execution_registry,
        [
            "model_catalog",
            "selected_models",
            "selected_runs",
            "selected_thresholds",
            "projection_eligibility",
            "model_reference",
        ],
    ),
    "run_locked_simca_validation_refit": (
        run_locked_simca_validation_refit,
        ["validation_executions"],
    ),
    "evaluate_locked_validation_predictions": (
        evaluate_locked_validation_predictions,
        [
            "validation_executions",
            "selected_thresholds",
            "object_predictions",
            "pixel_predictions",
        ],
    ),
    "build_locked_spatial_validation_outputs": (
        build_locked_spatial_validation_outputs,
        [
            "validation_executions",
            "selected_thresholds",
            "pixel_predictions",
            "image_db",
            "spatial_lock",
        ],
    ),
    "build_validation_guardrails": (
        build_validation_guardrails,
        ["validation_executions", "validation_metrics"],
    ),
}
for function_name, (function, expected_prefix) in expected_signatures.items():
    observed = list(inspect.signature(function).parameters)
    if observed[: len(expected_prefix)] != expected_prefix:
        raise RuntimeError(
            f"{function_name} still exposes an obsolete 04C signature: {observed}"
        )

vote_parameters = list(inspect.signature(build_pixel_vote_table).parameters)
if vote_parameters[:1] != ["pixel_predictions"] or "group_columns" not in vote_parameters:
    raise RuntimeError(
        "simca_thresholds_calibration.build_pixel_vote_table is not the migrated helper."
    )

# A signature-only edit of the spatial function is not sufficient: reject the
# known legacy implementation if its old body is still present.
spatial_source = inspect.getsource(build_locked_spatial_validation_outputs)
legacy_spatial_fragments = (
    "required_candidates =",
    'candidate["direct_2way_threshold"]',
    'groupby(["projection_config_id", "random_state"]',
)
if any(fragment in spatial_source for fragment in legacy_spatial_fragments):
    raise RuntimeError(
        "build_locked_spatial_validation_outputs still has the legacy 04C body. "
        "Replace the complete function, not only its signature."
    )

USE_WAVELENGTH_WINDOW = bool(expcfg.USE_WAVELENGTH_WINDOW)
RESULTS_TAG = (
    f"{int(expcfg.WAVELENGTH_WINDOW_MIN_NM)}_"
    f"{int(expcfg.WAVELENGTH_WINDOW_MAX_NM)}"
    if USE_WAVELENGTH_WINDOW
    else expcfg.DEFAULT_RESULTS_TAG
)

PROTOCOL_DIR = PROJECT_ROOT.joinpath(*expcfg.PROTOCOL_ARTIFACT_RELATIVE_DIR)

DB_H5_PATH = PROJECT_ROOT.joinpath(
    *expcfg.DATABASE_H5_RELATIVE_PATH
)

CALIBRATION_DIR = (
    PROJECT_ROOT
    / "results"
    / f"{expcfg.INTERNAL_CALIBRATION_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
)
DOMAIN_DIR = (
    PROJECT_ROOT
    / "results"
    / f"{expcfg.DOMAIN_SPATIAL_CALIBRATION_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
)
GRID_DIR = (
    PROJECT_ROOT
    / "results"
    / f"{expcfg.SIMCA_GRID_SEARCH_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
)
OPTUNA_DIR = (
    PROJECT_ROOT
    / "results"
    / f"{expcfg.SIMCA_OPTUNA_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
)
OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / f"{expcfg.SIMCA_CONCAT_REFIT_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATHS = {
    name: OUTPUT_DIR / filename
    for name, filename in expcfg.SIMCA_CONCAT_REFIT_OUTPUT_FILENAMES.items()
}
CHECKPOINT_DIR = OUTPUT_DIR / expcfg.SIMCA_CONCAT_REFIT_CHECKPOINT_DIRNAME

INPUT_03B = {
    key: CALIBRATION_DIR / expcfg.INTERNAL_CALIBRATION_OUTPUT_FILENAMES[key]
    for key in (
        "track_contracts",
        "model_catalog",
        "selected_models",
        "selected_runs",
        "selected_thresholds",
    )
}
INPUT_03B_MANIFEST = (
    CALIBRATION_DIR
    / expcfg.INTERNAL_CALIBRATION_OUTPUT_FILENAMES["checkpoint_manifest"]
)

INPUT_03C = {
    key: DOMAIN_DIR / expcfg.DOMAIN_SPATIAL_CALIBRATION_OUTPUT_FILENAMES[key]
    for key in (
        "projection_eligibility",
        "spatial_calibration_metrics",
        "fragment_size_classes",
        "spatial_postprocessing_lock",
        "audit_manifest",
    )
}

INPUT_04A = {
    key: GRID_DIR / expcfg.SIMCA_GRID_SEARCH_OUTPUT_FILENAMES[key]
    for key in ("model_reference", "audit_manifest")
}

# 04B is audit-only: only its manifest may be checked, and it never enters the
# execution registry or any scientific computation in 04C.
INPUT_04B_AUDIT = (
    OPTUNA_DIR / expcfg.SIMCA_OPTUNA_OUTPUT_FILENAMES["audit_manifest"]
)

VALIDATION_PLAN_HASH = hash_locked_validation_plan()
VALIDATION_EVALUATION_RULE_HASH = hash_locked_validation_evaluation_rule()

protocol_checks_df = verify_frozen_protocol(PROTOCOL_DIR, strict=True)
PROTOCOL_LOCK_PATH = PROTOCOL_DIR / expcfg.PROTOCOL_OUTPUT_FILENAMES["lock"]
protocol_lock = json.loads(PROTOCOL_LOCK_PATH.read_text(encoding="utf-8"))
PROTOCOL_HASH = str(protocol_lock["lock_sha256"])

print("Protocol:", expcfg.PROTOCOL_VERSION)
print("Global protocol hash:", PROTOCOL_HASH)
print("04C guardrail profile:", expcfg.SIMCA_CONCAT_REFIT_GUARDRAIL_PROFILE_ID)
print("04C evaluation rule:", expcfg.SIMCA_CONCAT_REFIT_EVALUATION_RULE_VERSION)
print("Validation plan hash:", VALIDATION_PLAN_HASH)
print("Evaluation-rule hash:", VALIDATION_EVALUATION_RULE_HASH)
print("Output:", OUTPUT_DIR)

Protocol: 8tracks_v5
Global protocol hash: af19c5f35d24a81a9c34d0a37c1f2776a31f5aecc70a9bfc1eecac42f8f91fa2
04C guardrail profile: 04c_fn_priority_strict_v1
04C evaluation rule: 04c_validation_metrics_v5_compact_ids_strict_guardrails
Validation plan hash: 3454161dd2955d1074d057acf0179f0b55f2c4b22dcf5ccc1656939815c505a5
Evaluation-rule hash: 8797ebd55d9ae4067d14a1cfc4846be0de495a1a41b4d798ac63cff347da4053
Output: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04C_simca_concat_refit_8tracks_v5_px_qc_v1


## C — Vérification de la lineage 03B → 03C → 04A (+ audit 04B)

La population scientifique de 04C vient **uniquement** de `selected_models.parquet` et `selected_runs.parquet` de 03B.  
Les manifests 03C et 04A sont vérifiés pour empêcher qu'un artefact homonyme mais provenant d'un autre run soit utilisé. Le manifest 04B, s'il existe, est seulement audité.

In [3]:
required_inputs = [
    INPUT_03B_MANIFEST,
    *INPUT_03B.values(),
    *INPUT_03C.values(),
    *INPUT_04A.values(),
]
missing_inputs = [str(path) for path in required_inputs if not path.is_file()]
if missing_inputs:
    raise FileNotFoundError(f"Missing upstream 04C inputs: {missing_inputs}")

manifest_03b = json.loads(INPUT_03B_MANIFEST.read_text(encoding="utf-8"))
validate_internal_calibration_manifest(
    manifest_03b,
    INPUT_03B,
    required_artifacts=tuple(INPUT_03B),
    protocol_hash=PROTOCOL_HASH,
)
artifact_hashes_03b = {
    str(entry["name"]): str(entry["sha256"])
    for entry in manifest_03b["artifacts"]
}

track_contracts_df = load_parquet(INPUT_03B["track_contracts"])
model_catalog_df = load_parquet(INPUT_03B["model_catalog"])
selected_models_df = load_parquet(INPUT_03B["selected_models"])
selected_runs_df = load_parquet(INPUT_03B["selected_runs"])
selected_threshold_rows_df = load_parquet(INPUT_03B["selected_thresholds"])

manifest_03c = json.loads(INPUT_03C["audit_manifest"].read_text(encoding="utf-8"))
if str(manifest_03c.get("protocol_version")) != str(expcfg.PROTOCOL_VERSION):
    raise RuntimeError("03C belongs to another protocol version.")
if str(manifest_03c.get("protocol_hash")) != PROTOCOL_HASH:
    raise RuntimeError("03C and the frozen protocol have different hashes.")
if manifest_03c.get("input_03b_manifest_sha256") != sha256_file(INPUT_03B_MANIFEST):
    raise RuntimeError("03C was not built from the current 03B manifest.")
if manifest_03c.get("natural_execution_key") != ["model_id", "random_state"]:
    raise RuntimeError("03C does not expose the canonical natural execution key.")

projection_eligibility_df = load_parquet(INPUT_03C["projection_eligibility"])
spatial_calibration_metrics_df = load_parquet(
    INPUT_03C["spatial_calibration_metrics"]
)
fragment_size_classes_df = load_parquet(INPUT_03C["fragment_size_classes"])
spatial_lock = json.loads(
    INPUT_03C["spatial_postprocessing_lock"].read_text(encoding="utf-8")
)

for key in (
    "projection_eligibility",
    "spatial_calibration_metrics",
    "fragment_size_classes",
):
    expected = str(
        manifest_03c["output_artifacts"][key]["sha256"]
    )
    observed = sha256_file(INPUT_03C[key])
    if observed != expected:
        raise RuntimeError(f"03C {key} hash mismatch.")

if str(manifest_03c["spatial_postprocessing_lock_sha256"]) != sha256_file(
    INPUT_03C["spatial_postprocessing_lock"]
):
    raise RuntimeError("03C spatial-lock file hash mismatch.")

verify_spatial_postprocessing_lock(
    spatial_lock,
    spatial_calibration_metrics_df,
    fragment_size_classes_df,
)

manifest_04a = json.loads(INPUT_04A["audit_manifest"].read_text(encoding="utf-8"))
if str(manifest_04a.get("protocol_hash")) != PROTOCOL_HASH:
    raise RuntimeError("04A and the frozen protocol have different hashes.")
if manifest_04a.get("input_03b_manifest_sha256") != sha256_file(INPUT_03B_MANIFEST):
    raise RuntimeError("04A was not built from the current 03B manifest.")
if manifest_04a.get("input_03c_manifest_sha256") != sha256_file(
    INPUT_03C["audit_manifest"]
):
    raise RuntimeError("04A was not built from the current 03C audit manifest.")
if manifest_04a.get("selection_authority") != "03B_selected_models":
    raise RuntimeError("04A does not preserve 03B selection authority.")
if bool(manifest_04a.get("selection_mutated", True)):
    raise RuntimeError("04A reports a mutation of the selected-model universe.")
if bool(manifest_04a.get("model_refit", True)):
    raise RuntimeError("04A unexpectedly reports a refit.")
if bool(manifest_04a.get("threshold_resuggestion", True)):
    raise RuntimeError("04A unexpectedly reports threshold resuggestion.")
if bool(manifest_04a.get("weighted_score_used", True)):
    raise RuntimeError("04A unexpectedly reports a weighted score.")
if manifest_04a.get("natural_execution_key") != ["model_id", "random_state"]:
    raise RuntimeError("04A does not preserve the canonical execution key.")

model_reference_df = load_parquet(INPUT_04A["model_reference"])
expected_model_reference_hash = str(
    manifest_04a["output_artifacts"]["model_reference"]["sha256"]
)
if sha256_file(INPUT_04A["model_reference"]) != expected_model_reference_hash:
    raise RuntimeError("04A selected-model reference hash mismatch.")

manifest_04b = None
if INPUT_04B_AUDIT.is_file():
    manifest_04b = json.loads(INPUT_04B_AUDIT.read_text(encoding="utf-8"))
    if str(manifest_04b.get("protocol_hash")) != PROTOCOL_HASH:
        raise RuntimeError("04B and the frozen protocol have different hashes.")
    if manifest_04b.get("input_03b_manifest_sha256") != sha256_file(
        INPUT_03B_MANIFEST
    ):
        raise RuntimeError("04B was not built from the current 03B manifest.")
    if manifest_04b.get("input_04a_manifest_sha256") != sha256_file(
        INPUT_04A["audit_manifest"]
    ):
        raise RuntimeError("04B was not built from the current 04A manifest.")
    if manifest_04b.get("selection_authority") != "03B_selected_models":
        raise RuntimeError("04B does not preserve 03B selection authority.")
    if bool(manifest_04b.get("selection_mutated", True)):
        raise RuntimeError("04B reports a forbidden selection mutation.")
    if bool(manifest_04b.get("model_refit", True)):
        raise RuntimeError("04B unexpectedly reports a model refit.")
    if bool(manifest_04b.get("threshold_resuggestion", True)):
        raise RuntimeError("04B unexpectedly reports threshold resuggestion.")
    if manifest_04b.get("downstream_selection_use") != "forbidden":
        raise RuntimeError("04B is not explicitly forbidden as a selection authority.")
    print("04B audit manifest verified; no 04B scientific table was loaded.")
else:
    print(
        "04B audit manifest not found. This does not block 04C because 04B "
        "is not a scientific input to the validation population."
    )

expected_track_ids = {f"E{index}" for index in range(1, 9)}
if set(track_contracts_df["track_id"].astype(str)) != expected_track_ids:
    raise RuntimeError("The 03B track contract must contain exactly E1-E8.")
if set(projection_eligibility_df["track_id"].astype(str)) != expected_track_ids:
    raise RuntimeError("03C eligibility must contain exactly E1-E8.")

if manifest_03c.get("spatial_selection_scope") != expcfg.SPATIAL_CALIBRATION_SELECTION_SCOPE:
    raise RuntimeError("03C manifest does not declare within-track spatial selection.")
if manifest_03c.get("spatial_selection_policy") != expcfg.SPATIAL_CALIBRATION_SELECTION_POLICY:
    raise RuntimeError("03C manifest has an unexpected spatial selection policy.")
if spatial_lock.get("selection_scope") != expcfg.SPATIAL_CALIBRATION_SELECTION_SCOPE:
    raise RuntimeError("04C requires the within-track 03C spatial lock.")
if spatial_lock.get("selection_policy") != expcfg.SPATIAL_CALIBRATION_SELECTION_POLICY:
    raise RuntimeError("04C received an unexpected spatial selection policy.")
if "selected_parameters" in spatial_lock or "selection_weighting" in spatial_lock:
    raise RuntimeError("Legacy global spatial-lock fields are forbidden.")

expected_spatial_tracks = set(
    projection_eligibility_df.loc[
        projection_eligibility_df["track_id"].astype(str).isin(
            expcfg.SPATIAL_CALIBRATION_PIXEL_TRACK_IDS
        )
        & projection_eligibility_df["eligibility_status"].astype(str).isin(
            expcfg.PROJECTION_DOMAIN_SPATIAL_SUPPORTED_STATUSES
        ),
        "track_id",
    ].astype(str)
)
locked_spatial_tracks = set(
    map(str, spatial_lock.get("selected_parameters_by_track", {}))
)
if locked_spatial_tracks != expected_spatial_tracks:
    raise RuntimeError(
        "03C spatial-lock tracks do not match supported pixel tracks: "
        f"expected={sorted(expected_spatial_tracks)}, "
        f"locked={sorted(locked_spatial_tracks)}."
    )

display(
    projection_eligibility_df[
        ["track_id", "eligibility_status", "eligibility_reason"]
    ].sort_values("track_id")
)

04B audit manifest verified; no 04B scientific table was loaded.


,track_id,eligibility_status,eligibility_reason
0,E1,eligible,all_predeclared_limits_satisfied
1,E2,eligible,all_predeclared_limits_satisfied
2,E3,unsupported_domain_shift,standardized_shift
3,E4,unsupported_domain_shift,standardized_shift
4,E5,eligible,all_predeclared_limits_satisfied
5,E6,eligible_with_warning,out_of_domain_rate;target_rejection_rate
6,E7,eligible,all_predeclared_limits_satisfied
7,E8,eligible_with_warning,out_of_domain_rate;target_rejection_rate


## D — Registre canonique des exécutions 04C

Aucun pool de candidats n'est reconstruit.  
`build_validation_execution_registry()` joint les identités déjà figées en 03B avec l'éligibilité 03C et le statut downstream 04A, sans ajouter ni retirer de modèle.

In [4]:
validation_executions_df, selected_thresholds_df = (
    build_validation_execution_registry(
        model_catalog=model_catalog_df,
        selected_models=selected_models_df,
        selected_runs=selected_runs_df,
        selected_thresholds=selected_threshold_rows_df,
        projection_eligibility=projection_eligibility_df,
        model_reference=model_reference_df,
        track_contracts=track_contracts_df,
    )
)

if list(validation_executions_df.columns) != list(
    expcfg.SIMCA_VALIDATION_EXECUTION_COLUMNS
):
    raise RuntimeError("Validation execution registry has an unexpected schema.")
if list(selected_thresholds_df.columns) != list(
    expcfg.INTERNAL_CALIBRATION_SELECTED_THRESHOLD_COLUMNS
):
    raise RuntimeError("Selected-threshold registry has an unexpected schema.")

run_key = ["model_id", "random_state"]
threshold_key = [*run_key, "decision_scope"]

if validation_executions_df.duplicated(run_key).any():
    raise RuntimeError("Duplicated natural execution key in 04C.")
if selected_thresholds_df.duplicated(threshold_key).any():
    raise RuntimeError("Duplicated natural decision-policy key in 04C.")
if validation_executions_df[["fit_id", "projection_id"]].isna().any().any():
    raise RuntimeError("Every validation execution must retain fit_id/projection_id.")

expected_scope_rows = []
for row in validation_executions_df.itertuples(index=False):
    expected_scope_rows.append(
        (str(row.model_id), int(row.random_state), "direct")
    )
    if str(row.projection_level) == "pixel_projection":
        expected_scope_rows.append(
            (str(row.model_id), int(row.random_state), "pixel_to_object")
        )

expected_scope_keys = set(expected_scope_rows)
observed_scope_keys = set(
    selected_thresholds_df[threshold_key].itertuples(index=False, name=None)
)
if observed_scope_keys != expected_scope_keys:
    raise RuntimeError("Selected threshold scopes do not match validation executions.")

legacy_identifier_columns = {
    "validation_candidate_id",
    "calibration_id",
    "domain_config_id",
    "evaluation_config_id",
    "data_config_id",
    "fit_config_id",
    "projection_config_id",
    "run_id",
}
leaked = sorted(
    legacy_identifier_columns.intersection(validation_executions_df.columns)
)
if leaked:
    raise RuntimeError(f"04C execution registry leaks legacy identifiers: {leaked}")

EXECUTION_REGISTRY_HASH = sha256_dataframe(
    validation_executions_df.reindex(
        columns=expcfg.SIMCA_VALIDATION_EXECUTION_COLUMNS
    )
)
SELECTED_THRESHOLDS_HASH = sha256_dataframe(
    selected_thresholds_df.reindex(
        columns=expcfg.INTERNAL_CALIBRATION_SELECTED_THRESHOLD_COLUMNS
    )
)

execution_summary = (
    validation_executions_df.groupby(
        ["track_id", "eligibility_status", "downstream_status"],
        as_index=False,
        dropna=False,
        sort=False,
    )
    .agg(
        n_models=("model_id", "nunique"),
        n_executions=("model_id", "size"),
        n_fits=("fit_id", "nunique"),
        n_projections=("projection_id", "nunique"),
    )
    .sort_values("track_id")
)
display(execution_summary)

print("Selected models:", validation_executions_df["model_id"].nunique())
print("Selected executions:", len(validation_executions_df))
print("Unique fits:", validation_executions_df["fit_id"].nunique())
print("Unique projections:", validation_executions_df["projection_id"].nunique())
print("Execution registry hash:", EXECUTION_REGISTRY_HASH)
print("Selected thresholds hash:", SELECTED_THRESHOLDS_HASH)

,track_id,eligibility_status,downstream_status,n_models,n_executions,n_fits,n_projections
7,E1,eligible,supported,2,2,1,2
0,E2,eligible,supported,8,8,8,8
5,E3,unsupported_domain_shift,diagnostic_only,1,1,1,1
2,E4,unsupported_domain_shift,diagnostic_only,1,1,1,1
3,E5,eligible,supported,1,3,3,3
1,E6,eligible_with_warning,supported,18,40,29,40
6,E7,eligible,supported,1,3,3,3
4,E8,eligible_with_warning,supported,7,21,12,21


Selected models: 39
Selected executions: 79
Unique fits: 46
Unique projections: 76
Execution registry hash: 333295a25d7a0eeef69c47816e1be0ac7c9cc2218fce3c346a92982deb64742e
Selected thresholds hash: f4ca411176747218f26e7829faf132657dcb03f311cb45fec2a38d4904ccb8a8


## E — Chargement strict des batches 1–3

Le batch 3 est ouvert **seulement après** validation de toute la lineage amont et après calcul des hashes du plan et du registre d'exécution. Le batch 4 est interdit.

In [5]:
if not expcfg.SIMCA_CONCAT_REFIT_RUN:
    raise RuntimeError(
        "SIMCA_CONCAT_REFIT_RUN=False: enable it to produce the new 04C outputs."
    )

allowed_batches = tuple(
    sorted(
        set(
            tuple(expcfg.SIMCA_CONCAT_REFIT_TRAIN_BATCHES)
            + tuple(expcfg.SIMCA_CONCAT_REFIT_PROJECTION_BATCHES)
        )
    )
)
forbidden_batches = set(map(int, expcfg.SIMCA_CONCAT_REFIT_FORBIDDEN_BATCHES))
if set(map(int, allowed_batches)).intersection(forbidden_batches):
    raise RuntimeError("A forbidden batch entered the 04C load contract.")

if not DB_H5_PATH.is_file():
    raise FileNotFoundError(DB_H5_PATH)

DATABASE_SHA256 = sha256_file(DB_H5_PATH)

object_db, image_db = load_nir_uco_h5(
    DB_H5_PATH,
    reconstruct_heavy_object_arrays=(
        expcfg.SIMCA_CONCAT_REFIT_RECONSTRUCT_HEAVY_OBJECT_ARRAYS
    ),
    batches=allowed_batches,
)

loaded_batches = {
    int(record["batch"])
    for record in object_db.values()
    if record.get("batch") is not None
}
if not loaded_batches.issubset(set(map(int, allowed_batches))):
    raise RuntimeError(f"Unexpected HDF5 batches: {sorted(loaded_batches)}")
if loaded_batches.intersection(forbidden_batches):
    raise RuntimeError("Batch 4 was loaded in notebook 04C.")

required_loaded_batches = set(
    map(
        int,
        tuple(expcfg.SIMCA_CONCAT_REFIT_TRAIN_BATCHES)
        + tuple(expcfg.SIMCA_CONCAT_REFIT_PROJECTION_BATCHES),
    )
)
if loaded_batches != required_loaded_batches:
    raise RuntimeError(
        "04C did not load exactly the required train+validation batches: "
        f"observed={sorted(loaded_batches)}, "
        f"expected={sorted(required_loaded_batches)}."
    )

if USE_WAVELENGTH_WINDOW:
    object_db, image_db, wavelengths, _ = select_wavelength_range_from_database(
        object_db=object_db,
        image_db=image_db,
        min_nm=expcfg.WAVELENGTH_WINDOW_MIN_NM,
        max_nm=expcfg.WAVELENGTH_WINDOW_MAX_NM,
    )
else:
    wavelengths = np.asarray(
        next(iter(object_db.values()))["wavelengths"],
        dtype=float,
    )

print("Database:", DB_H5_PATH)
print("Database SHA-256:", DATABASE_SHA256)
print("Loaded batches:", sorted(loaded_batches))
print("Spectral bands:", len(wavelengths))

Database: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\HSI Data\processed\nir_uco_database_8tracks_v5.h5
Database SHA-256: bb16004c1fccab52e0203313fbb96db97071a56e04ed9d352d22810d35bf832b
Loaded batches: [1, 2, 3]
Spectral bands: 61


## F — Refit batches 1–2 et projection continue du batch 3

Chaque `fit_id` est ajusté une seule fois à l'intérieur de son groupe de calcul et chaque `projection_id` produit une seule table continue. Les politiques 2-way/3-way ne sont appliquées qu'ensuite, afin de ne pas dupliquer les grandes tables pixels.

In [6]:
checkpoint_context = {
    "protocol_hash": PROTOCOL_HASH,
    "validation_plan_hash": VALIDATION_PLAN_HASH,
    "execution_registry_hash": EXECUTION_REGISTRY_HASH,
    "database_sha256": DATABASE_SHA256,
}
refit_kwargs = {
    "wavelengths": wavelengths,
    "train_batches": expcfg.SIMCA_CONCAT_REFIT_TRAIN_BATCHES,
    "projection_batches": expcfg.SIMCA_CONCAT_REFIT_PROJECTION_BATCHES,
    "target_class": expcfg.TARGET_CLASS,
    "non_target_label": expcfg.NON_TARGET_LABEL,
    "border_width": expcfg.SIMCA_CONCAT_REFIT_BORDER_WIDTH,
    "verbose": expcfg.SIMCA_CONCAT_REFIT_VERBOSE,
}

if expcfg.SIMCA_CONCAT_REFIT_CHECKPOINT_ENABLED:
    refit_outputs = run_locked_simca_validation_refit_checkpointed(
        validation_executions_df,
        object_db=object_db,
        checkpoint_dir=CHECKPOINT_DIR,
        checkpoint_context=checkpoint_context,
        resume=expcfg.SIMCA_CONCAT_REFIT_RESUME_FROM_CHECKPOINT,
        **refit_kwargs,
    )
    CHECKPOINT_RUN_DIR = Path(refit_outputs["checkpoint_run_dir"])
else:
    refit_outputs = run_locked_simca_validation_refit(
        validation_executions_df,
        object_db=object_db,
        **refit_kwargs,
    )
    CHECKPOINT_RUN_DIR = None

validation_object_predictions_df = refit_outputs["object_predictions"]
validation_pixel_predictions_df = refit_outputs["pixel_predictions"]
technical_events_df = refit_outputs["technical_events"]

prediction_contracts = (
    (
        "object_predictions",
        validation_object_predictions_df,
        expcfg.SIMCA_VALIDATION_OBJECT_PREDICTION_COLUMNS,
        ["projection_id", "source_image", "object_id"],
    ),
    (
        "pixel_predictions",
        validation_pixel_predictions_df,
        expcfg.SIMCA_VALIDATION_PIXEL_PREDICTION_COLUMNS,
        ["projection_id", "source_image", "object_id", "row", "col"],
    ),
)
for name, frame, schema, natural_key in prediction_contracts:
    if list(frame.columns) != list(schema):
        raise RuntimeError(f"{name} has an unexpected compact schema.")
    if len(frame) and frame.duplicated(natural_key).any():
        raise RuntimeError(f"{name} duplicates its natural observation key.")
    if len(frame):
        observed = set(pd.to_numeric(frame["batch"], errors="raise").astype(int))
        if observed != set(map(int, expcfg.SIMCA_CONCAT_REFIT_PROJECTION_BATCHES)):
            raise RuntimeError(
                f"{name} contains batches other than the locked batch 3."
            )
    leaked = sorted(legacy_identifier_columns.intersection(frame.columns))
    if leaked:
        raise RuntimeError(f"{name} leaks legacy identifiers: {leaked}")

if list(technical_events_df.columns) != list(
    expcfg.SIMCA_VALIDATION_TECHNICAL_EVENT_COLUMNS
):
    raise RuntimeError("technical_events has an unexpected schema.")

display(
    pd.DataFrame(
        {
            "table": [
                "object_predictions",
                "pixel_predictions",
                "technical_events",
            ],
            "rows": [
                len(validation_object_predictions_df),
                len(validation_pixel_predictions_df),
                len(technical_events_df),
            ],
        }
    )
)
if len(technical_events_df):
    display(
        technical_events_df.groupby(
            ["stage", "error_type"],
            as_index=False,
            dropna=False,
        ).size()
    )

,table,rows
0,object_predictions,5724
1,pixel_predictions,156561
2,technical_events,0


## G — Application des politiques de décision 03B

Les seuils de `selected_thresholds.parquet` sont appliqués **sans réajustement** :

- `direct` pour toutes les projections ;
- `pixel_to_object` uniquement pour les tracks à projection pixel.

Le vote pixel→objet réutilise exactement la définition 03B (`simca_margin >= 0` avant agrégation du ratio), via le helper partagé du module de calibration des seuils.

In [7]:
validation_metrics_df = evaluate_locked_validation_predictions(
    validation_executions_df,
    selected_thresholds_df,
    validation_object_predictions_df,
    validation_pixel_predictions_df,
    technical_events=technical_events_df,
)

if list(validation_metrics_df.columns) != list(
    expcfg.SIMCA_VALIDATION_METRIC_COLUMNS
):
    raise RuntimeError("validation_metrics has an unexpected long schema.")

metric_key = [
    "model_id",
    "random_state",
    "decision_scope",
    "map_variant",
    "aggregation_level",
    "group_id",
    "metric",
]
if len(validation_metrics_df) and validation_metrics_df.duplicated(metric_key).any():
    raise RuntimeError("validation_metrics duplicates its natural long-form key.")

aggregate_forbidden_ids = {
    *legacy_identifier_columns,
    "fit_id",
    "projection_id",
}
leaked = sorted(aggregate_forbidden_ids.intersection(validation_metrics_df.columns))
if leaked:
    raise RuntimeError(f"validation_metrics leaks technical identifiers: {leaked}")

observed_metric_scopes = set(
    validation_metrics_df[
        ["model_id", "random_state", "decision_scope"]
    ].drop_duplicates().itertuples(index=False, name=None)
)
if observed_metric_scopes != expected_scope_keys:
    missing = sorted(expected_scope_keys - observed_metric_scopes)
    extra = sorted(observed_metric_scopes - expected_scope_keys)
    raise RuntimeError(
        "Validation metric scope coverage is incomplete: "
        f"missing={missing[:10]}, extra={extra[:10]}."
    )

overall_metrics = validation_metrics_df.loc[
    validation_metrics_df["aggregation_level"].eq("overall")
    & validation_metrics_df["group_id"].eq("all")
].copy()

metrics_to_show = (
    "target_miss_rate",
    "false_accept_rate",
    "uncertain_rate",
    "coverage_rate",
    "balanced_accuracy",
    "decided_balanced_accuracy",
    "macro_image_target_miss_rate",
    "macro_image_false_accept_rate",
    "macro_image_uncertain_rate",
)
overall_display = (
    overall_metrics.loc[
        overall_metrics["metric"].isin(metrics_to_show),
        [
            "model_id",
            "random_state",
            "track_id",
            "decision_scope",
            "status",
            "metric",
            "value",
        ],
    ]
    .pivot_table(
        index=[
            "model_id",
            "random_state",
            "track_id",
            "decision_scope",
            "status",
        ],
        columns="metric",
        values="value",
        aggfunc="first",
    )
    .reset_index()
)
display(overall_display)

display(
    overall_metrics[
        ["model_id", "random_state", "track_id", "decision_scope", "status"]
    ]
    .drop_duplicates()
    .groupby(["track_id", "decision_scope", "status"], as_index=False)
    .size()
)

metric,model_id,random_state,track_id,decision_scope,status,balanced_accuracy,coverage_rate,decided_balanced_accuracy,false_accept_rate,macro_image_false_accept_rate,macro_image_target_miss_rate,macro_image_uncertain_rate,target_miss_rate,uncertain_rate
0,model_00b01181ce99fe157681,0,E2,direct,calculable,NaN,0.657407,0.750000,0.163636,0.163636,0.000000,0.336364,0.000000,0.342593
1,model_027bc4d5417f5fea1c7d,0,E6,direct,calculable,NaN,0.981481,0.811321,0.363636,0.363636,0.000000,0.018182,0.000000,0.018519
2,model_027bc4d5417f5fea1c7d,1,E6,direct,calculable,0.818182,1.000000,0.818182,0.363636,0.363636,0.000000,0.000000,0.000000,0.000000
3,model_027bc4d5417f5fea1c7d,2,E6,direct,calculable,NaN,0.972222,0.759615,0.454545,0.454545,0.000000,0.027273,0.000000,0.027778
4,model_0326bb57ce83ca64ed15,0,E6,direct,calculable,NaN,0.888889,0.883721,0.181818,0.181818,0.000000,0.109091,0.000000,0.111111
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,model_f662665e0f4a7a7c4783,0,E8,pixel_to_object,calculable,NaN,0.824074,0.791667,0.272727,0.272727,0.000000,0.172727,0.000000,0.175926
101,model_f662665e0f4a7a7c4783,1,E8,direct,calculable,NaN,0.667107,0.755150,0.226895,0.226895,0.008769,0.320088,0.008769,0.332893
102,model_f662665e0f4a7a7c4783,1,E8,pixel_to_object,calculable,NaN,0.842593,0.671053,0.454545,0.454545,0.000000,0.154545,0.000000,0.157407
103,model_f662665e0f4a7a7c4783,2,E8,direct,calculable,NaN,0.631262,0.863559,0.119535,0.119535,0.014093,0.357950,0.014093,0.368738


,track_id,decision_scope,status,size
0,E1,direct,calculable,2
1,E2,direct,calculable,8
2,E3,direct,calculable,1
3,E3,pixel_to_object,calculable,1
4,E4,direct,calculable,1
5,E4,pixel_to_object,calculable,1
6,E5,direct,calculable,3
7,E6,direct,calculable,40
8,E7,direct,calculable,3
9,E7,pixel_to_object,calculable,3


## H — Reconstruction spatiale verrouillée

Seules les décisions pixel `direct` des tracks **pixel-projection supportés** servent aux cartes spatiales. Chaque track utilise exclusivement les paramètres verrouillés pour son propre `track_id` :

`projection continue → décision pixel directe → raw_target_mask → morphologie verrouillée 03C → postprocessed_target_mask`.

Le scope `pixel_to_object` est une décision au niveau objet et **n'intervient jamais** dans les cartes.

Les tracks `object_projection` ne produisent pas de carte de décision pixel : aucune morphologie 2D n'y est donc appliquée.

In [8]:
spatial_outputs = build_locked_spatial_validation_outputs(
    validation_executions_df,
    selected_thresholds_df,
    validation_pixel_predictions_df,
    image_db,
    spatial_lock,
)

pixel_maps_manifest_df = spatial_outputs["pixel_maps_manifest"]
spatial_components_df = spatial_outputs["spatial_components"]
spatial_component_metrics_df = spatial_outputs["spatial_component_metrics"]

spatial_contracts = (
    (
        "pixel_maps_manifest",
        pixel_maps_manifest_df,
        expcfg.SIMCA_PIXEL_MAP_MANIFEST_COLUMNS,
    ),
    (
        "spatial_components",
        spatial_components_df,
        expcfg.SIMCA_SPATIAL_COMPONENT_COLUMNS,
    ),
    (
        "spatial_component_metrics",
        spatial_component_metrics_df,
        expcfg.SIMCA_SPATIAL_COMPONENT_METRIC_COLUMNS,
    ),
)
for name, frame, schema in spatial_contracts:
    if list(frame.columns) != list(schema):
        raise RuntimeError(f"{name} has an unexpected compact schema.")
    leaked = sorted(aggregate_forbidden_ids.intersection(frame.columns))
    if leaked:
        raise RuntimeError(f"{name} leaks technical/legacy identifiers: {leaked}")

observed_spatial_tracks = set(
    pixel_maps_manifest_df["track_id"].astype(str)
) if len(pixel_maps_manifest_df) else set()
if observed_spatial_tracks != expected_spatial_tracks:
    raise RuntimeError(
        "04C spatial outputs do not cover exactly the supported pixel tracks: "
        f"expected={sorted(expected_spatial_tracks)}, "
        f"observed={sorted(observed_spatial_tracks)}."
    )

if len(pixel_maps_manifest_df):
    map_key = ["track_id", "model_id", "random_state", "source_image"]
    if pixel_maps_manifest_df.duplicated(map_key).any():
        raise RuntimeError("pixel_maps_manifest duplicates its natural map key.")
    if not pixel_maps_manifest_df["truth_level"].astype(str).eq(
        expcfg.SIMCA_CONCAT_REFIT_TRUTH_SOURCE
    ).all():
        raise RuntimeError("Spatial truth does not match the locked 04C truth source.")

if len(spatial_component_metrics_df):
    spatial_metric_key = [
        "track_id",
        "model_id",
        "random_state",
        "source_image",
        "aggregation_level",
        "map_variant",
    ]
    if spatial_component_metrics_df.duplicated(spatial_metric_key).any():
        raise RuntimeError(
            "spatial_component_metrics duplicates its natural metric key."
        )
    observed_variants = set(
        spatial_component_metrics_df["map_variant"].astype(str)
    )
    if not {"raw", "locked_postprocessed"}.issubset(observed_variants):
        raise RuntimeError(
            "Spatial validation requires raw and locked_postprocessed maps."
        )

    spatial_overall_display = spatial_component_metrics_df.loc[
        spatial_component_metrics_df["aggregation_level"].eq("overall"),
        [
            "model_id",
            "random_state",
            "track_id",
            "map_variant",
            "dice",
            "iou",
            "pixel_recall",
            "component_recall",
            "smallest_fragment_recall",
        ],
    ].sort_values(["track_id", "model_id", "random_state", "map_variant"])
    display(spatial_overall_display)
else:
    print(
        "No spatial metrics were produced. Check technical_events if pixel "
        "projections were expected."
    )

,model_id,random_state,track_id,map_variant,dice,iou,pixel_recall,component_recall,smallest_fragment_recall
0,model_56e46be423e049e843e7,0,E7,locked_postprocessed,0.778825,0.637766,0.983714,1.0,1.0
1,model_56e46be423e049e843e7,0,E7,raw,0.741239,0.588864,0.983714,1.0,1.0
6,model_56e46be423e049e843e7,1,E7,locked_postprocessed,0.771677,0.628236,0.988099,1.0,1.0
7,model_56e46be423e049e843e7,1,E7,raw,0.740002,0.587305,0.988099,1.0,1.0
12,model_56e46be423e049e843e7,2,E7,locked_postprocessed,0.858135,0.751521,0.967116,1.0,1.0
...,...,...,...,...,...,...,...,...,...
127,model_f662665e0f4a7a7c4783,0,E8,raw,0.885762,0.794948,0.980572,1.0,1.0
132,model_f662665e0f4a7a7c4783,1,E8,locked_postprocessed,0.866451,0.764369,0.990819,1.0,1.0
133,model_f662665e0f4a7a7c4783,1,E8,raw,0.868649,0.767798,0.990113,1.0,1.0
138,model_f662665e0f4a7a7c4783,2,E8,locked_postprocessed,0.913919,0.841483,0.983506,1.0,1.0


## I — Guardrails stricts 04C

Les guardrails sont appliqués séparément à chaque `(model_id, random_state, decision_scope)`.

- pour les modèles `supported`, les contrôles prescrits sont **bloquants** ;
- pour les modèles `diagnostic_only`, les mêmes diagnostics sont calculés mais ne peuvent pas devenir une sélection scientifique ;
- aucune moyenne ni aucun rang entre tracks n'est utilisé ;
- le rappel de la plus petite classe de fragments reste diagnostique tant qu'aucun seuil indépendant n'a été préspécifié.

In [9]:
validation_guardrails_df = build_validation_guardrails(
    validation_executions_df,
    validation_metrics_df,
    spatial_component_metrics=spatial_component_metrics_df,
)

if list(validation_guardrails_df.columns) != list(
    expcfg.SIMCA_VALIDATION_GUARDRAIL_COLUMNS
):
    raise RuntimeError("validation_guardrails has an unexpected schema.")

guardrail_key = [
    "model_id",
    "random_state",
    "decision_scope",
    "scope",
    "metric",
]
if len(validation_guardrails_df) and validation_guardrails_df.duplicated(
    guardrail_key
).any():
    raise RuntimeError("validation_guardrails duplicates its natural key.")

leaked = sorted(aggregate_forbidden_ids.intersection(validation_guardrails_df.columns))
if leaked:
    raise RuntimeError(f"validation_guardrails leaks technical identifiers: {leaked}")

if any(
    "score" in str(column).lower()
    for column in validation_guardrails_df.columns
):
    raise RuntimeError("A composite score entered the 04C guardrail table.")

diagnostic_rows = validation_guardrails_df["downstream_status"].astype(str).eq(
    "diagnostic_only"
)
if validation_guardrails_df.loc[diagnostic_rows, "is_blocking"].astype(bool).any():
    raise RuntimeError("A diagnostic-only model received a blocking guardrail.")

scientific_checks = validation_guardrails_df.loc[
    validation_guardrails_df["check_status"].isin(["pass", "fail"])
]
if len(scientific_checks):
    observed = pd.to_numeric(
        scientific_checks["observed_value"], errors="coerce"
    ).to_numpy(dtype=float)
    if not np.isfinite(observed).all():
        raise RuntimeError(
            "A scientific pass/fail guardrail contains a non-finite metric."
        )

status_table = (
    validation_guardrails_df[
        [
            "model_id",
            "random_state",
            "track_id",
            "decision_scope",
            "candidate_status",
        ]
    ]
    .drop_duplicates()
    .groupby(
        ["track_id", "decision_scope", "candidate_status"],
        as_index=False,
        dropna=False,
    )
    .size()
    .rename(columns={"size": "n_execution_scopes"})
)
display(status_table)

blocking_summary = (
    validation_guardrails_df.loc[
        validation_guardrails_df["is_blocking"].astype(bool)
    ]
    .groupby(
        ["track_id", "decision_scope", "check_status"],
        as_index=False,
        dropna=False,
    )
    .size()
    .rename(columns={"size": "n_checks"})
)
display(blocking_summary)

,track_id,decision_scope,candidate_status,n_execution_scopes
0,E1,direct,calculable_but_not_acceptable,2
1,E2,direct,calculable_but_not_acceptable,5
2,E2,direct,pass,3
3,E3,direct,diagnostic_only,1
4,E3,pixel_to_object,diagnostic_only,1
5,E4,direct,diagnostic_only,1
6,E4,pixel_to_object,diagnostic_only,1
7,E5,direct,pass,3
8,E6,direct,calculable_but_not_acceptable,31
9,E6,direct,pass,9


,track_id,decision_scope,check_status,n_checks
0,E1,direct,fail,2
1,E1,direct,pass,8
2,E2,direct,fail,10
3,E2,direct,pass,46
4,E5,direct,pass,15
5,E6,direct,fail,49
6,E6,direct,pass,231
7,E7,direct,fail,7
8,E7,direct,pass,8
9,E7,pixel_to_object,fail,1


## J — Persistance et manifest 04C

Les sorties enregistrées utilisent uniquement les identifiants canoniques nécessaires.  
Le `validation_protocol.json` constitue la frontière de lineage pour le notebook 05 : il enregistre les hashes du protocole global, du contrat 04C, du registre d'exécution, des seuils 03B, des entrées et des sorties.

In [10]:
tables_to_save = {
    "object_predictions": validation_object_predictions_df.reindex(
        columns=expcfg.SIMCA_VALIDATION_OBJECT_PREDICTION_COLUMNS
    ),
    "pixel_predictions": validation_pixel_predictions_df.reindex(
        columns=expcfg.SIMCA_VALIDATION_PIXEL_PREDICTION_COLUMNS
    ),
    "metrics": validation_metrics_df.reindex(
        columns=expcfg.SIMCA_VALIDATION_METRIC_COLUMNS
    ),
    "pixel_maps_manifest": pixel_maps_manifest_df.reindex(
        columns=expcfg.SIMCA_PIXEL_MAP_MANIFEST_COLUMNS
    ),
    "spatial_components": spatial_components_df.reindex(
        columns=expcfg.SIMCA_SPATIAL_COMPONENT_COLUMNS
    ),
    "spatial_component_metrics": spatial_component_metrics_df.reindex(
        columns=expcfg.SIMCA_SPATIAL_COMPONENT_METRIC_COLUMNS
    ),
    "guardrails": validation_guardrails_df.reindex(
        columns=expcfg.SIMCA_VALIDATION_GUARDRAIL_COLUMNS
    ),
    "technical_events": technical_events_df.reindex(
        columns=expcfg.SIMCA_VALIDATION_TECHNICAL_EVENT_COLUMNS
    ),
}

# One final identifier-leak audit before persistence.
for name, table in tables_to_save.items():
    if name in {"object_predictions", "pixel_predictions"}:
        forbidden = legacy_identifier_columns
    elif name == "technical_events":
        forbidden = legacy_identifier_columns
    else:
        forbidden = aggregate_forbidden_ids
    leaked = sorted(set(forbidden).intersection(table.columns))
    if leaked:
        raise RuntimeError(f"{name} contains forbidden identifiers: {leaked}")

for name, table in tables_to_save.items():
    save_parquet(table, OUTPUT_PATHS[name])

output_sha256 = {
    name: sha256_file(OUTPUT_PATHS[name])
    for name in tables_to_save
}

UPSTREAM_SHA256 = {
    "03B.checkpoint_manifest": sha256_file(INPUT_03B_MANIFEST),
    **{
        f"03B.{key}": sha256_file(path)
        for key, path in INPUT_03B.items()
    },
    **{
        f"03C.{key}": sha256_file(path)
        for key, path in INPUT_03C.items()
    },
    **{
        f"04A.{key}": sha256_file(path)
        for key, path in INPUT_04A.items()
    },
}
if INPUT_04B_AUDIT.is_file():
    UPSTREAM_SHA256["04B.audit_manifest"] = sha256_file(INPUT_04B_AUDIT)

execution_scope_status = (
    validation_guardrails_df[
        [
            "model_id",
            "random_state",
            "decision_scope",
            "candidate_status",
        ]
    ]
    .drop_duplicates()
)
status_counts = {
    str(status): int(count)
    for status, count in execution_scope_status["candidate_status"]
    .value_counts(dropna=False)
    .items()
}

previous_protocol = {}
if OUTPUT_PATHS["protocol"].is_file():
    previous_protocol = json.loads(
        OUTPUT_PATHS["protocol"].read_text(encoding="utf-8")
    )

protocol = {
    "notebook": "04C_simca_concat_refit",
    "tasks": [31, 32, 33],
    "results_tag": RESULTS_TAG,
    "protocol_version": str(expcfg.PROTOCOL_VERSION),
    "schema_version": str(expcfg.RESULTS_SCHEMA_VERSION),
    "protocol_hash": PROTOCOL_HASH,
    "contract_role": "batch3_validation_child_contract_within_8tracks_v5",
    "selection_authority": "03B_selected_models",
    "selection_mutated": False,
    "execution_population_source": "03B_selected_models_and_selected_runs",
    "execution_policy": str(expcfg.SIMCA_CONCAT_REFIT_EXECUTION_POLICY),
    "model_refit": True,
    "model_refit_train_batches": list(
        map(int, expcfg.SIMCA_CONCAT_REFIT_TRAIN_BATCHES)
    ),
    "projection_batches": list(
        map(int, expcfg.SIMCA_CONCAT_REFIT_PROJECTION_BATCHES)
    ),
    "forbidden_batches": list(
        map(int, expcfg.SIMCA_CONCAT_REFIT_FORBIDDEN_BATCHES)
    ),
    "batch4_loaded": False,
    "threshold_resuggestion": False,
    "threshold_policy": "fixed_selected_03B_thresholds_no_batch3_recalibration",
    "weighted_score_used": False,
    "cross_track_selection_or_ranking": False,
    "natural_execution_key": ["model_id", "random_state"],
    "continuous_projection_key": ["projection_id"],
    "decision_policy_key": [
        "model_id",
        "random_state",
        "decision_scope",
    ],
    "validation_plan_hash": VALIDATION_PLAN_HASH,
    "execution_registry_sha256": EXECUTION_REGISTRY_HASH,
    "selected_thresholds_sha256": SELECTED_THRESHOLDS_HASH,
    "validation_evaluation_rule_version": str(
        expcfg.SIMCA_CONCAT_REFIT_EVALUATION_RULE_VERSION
    ),
    "validation_evaluation_rule_hash": VALIDATION_EVALUATION_RULE_HASH,
    "guardrail_profile_id": str(
        expcfg.SIMCA_CONCAT_REFIT_GUARDRAIL_PROFILE_ID
    ),
    "guardrail_limits": expcfg.SIMCA_CONCAT_REFIT_GUARDRAIL_LIMITS,
    "guardrail_check_specs": expcfg.SIMCA_CONCAT_REFIT_GUARDRAIL_CHECK_SPECS,
    "guardrail_amendment": amendment,
    "supported_guardrails_blocking": True,
    "diagnostic_only_guardrails_blocking": False,
    "smallest_fragment_guardrail": (
        "diagnostic_only_no_prespecified_threshold"
        if expcfg.SIMCA_CONCAT_REFIT_SMALLEST_FRAGMENT_RECALL_MIN is None
        else float(expcfg.SIMCA_CONCAT_REFIT_SMALLEST_FRAGMENT_RECALL_MIN)
    ),
    "spatial_decision_scope": "direct_only",
    "spatial_application_policy": (
        "supported_pixel_projection_tracks_only_with_track_specific_lock"
    ),
    "spatial_selection_scope": str(spatial_lock["selection_scope"]),
    "spatial_selection_policy": str(spatial_lock["selection_policy"]),
    "spatial_track_ids": list(map(str, spatial_lock["spatial_track_ids"])),
    "spatial_parameters_by_track": spatial_lock["selected_parameters_by_track"],
    "spatial_lock_payload_sha256": str(spatial_lock["lock_sha256"]),
    "spatial_lock_file_sha256": sha256_file(
        INPUT_03C["spatial_postprocessing_lock"]
    ),
    "spatial_truth": str(expcfg.SIMCA_CONCAT_REFIT_TRUTH_SOURCE),
    "04a_role": "selected_model_reference_and_downstream_status_only",
    "04b_role": "audit_only_never_used_for_execution_population",
    "04b_manifest_verified": bool(manifest_04b is not None),
    "database_sha256": DATABASE_SHA256,
    "checkpoint_context": checkpoint_context,
    "checkpoint_run_dir": (
        None if CHECKPOINT_RUN_DIR is None else str(CHECKPOINT_RUN_DIR)
    ),
    "counts": {
        "models": int(validation_executions_df["model_id"].nunique()),
        "executions": int(len(validation_executions_df)),
        "fits": int(validation_executions_df["fit_id"].nunique()),
        "projections": int(validation_executions_df["projection_id"].nunique()),
        "decision_scopes": int(len(expected_scope_keys)),
        "object_prediction_rows": int(len(validation_object_predictions_df)),
        "pixel_prediction_rows": int(len(validation_pixel_predictions_df)),
        "technical_events": int(len(technical_events_df)),
        "pixel_maps": int(len(pixel_maps_manifest_df)),
        "spatial_component_rows": int(len(spatial_components_df)),
        "guardrail_rows": int(len(validation_guardrails_df)),
    },
    "execution_scope_status_counts": status_counts,
    "input_sha256": UPSTREAM_SHA256,
    "output_sha256": output_sha256,
    "supersedes_evaluation_rule_hash": (
        previous_protocol.get("validation_evaluation_rule_hash")
        if previous_protocol
        and previous_protocol.get("validation_evaluation_rule_hash")
        != VALIDATION_EVALUATION_RULE_HASH
        else None
    ),
}

OUTPUT_PATHS["protocol"].write_text(
    json.dumps(
        protocol,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

# Read-back integrity checks after every file has been written.
for name, table in tables_to_save.items():
    if sha256_file(OUTPUT_PATHS[name]) != output_sha256[name]:
        raise RuntimeError(f"{name} changed immediately after persistence.")
    persisted = load_parquet(OUTPUT_PATHS[name])
    if list(persisted.columns) != list(table.columns):
        raise RuntimeError(f"{name} schema changed after persistence.")

print("04C completed and persisted.")
print("Validation protocol:", OUTPUT_PATHS["protocol"])
print("Evaluation-rule hash:", VALIDATION_EVALUATION_RULE_HASH)
display(status_table)

04C completed and persisted.
Validation protocol: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04C_simca_concat_refit_8tracks_v5_px_qc_v1\validation_protocol.json
Evaluation-rule hash: 8797ebd55d9ae4067d14a1cfc4846be0de495a1a41b4d798ac63cff347da4053


,track_id,decision_scope,candidate_status,n_execution_scopes
0,E1,direct,calculable_but_not_acceptable,2
1,E2,direct,calculable_but_not_acceptable,5
2,E2,direct,pass,3
3,E3,direct,diagnostic_only,1
4,E3,pixel_to_object,diagnostic_only,1
5,E4,direct,diagnostic_only,1
6,E4,pixel_to_object,diagnostic_only,1
7,E5,direct,pass,3
8,E6,direct,calculable_but_not_acceptable,31
9,E6,direct,pass,9


## Contrat de sortie

- `validation_object_predictions.parquet` : projections objet continues, indexées par `projection_id`.
- `validation_pixel_predictions.parquet` : projections pixel continues, indexées par `projection_id` et coordonnées.
- `validation_metrics.parquet` : métriques longues par `(model_id, random_state, decision_scope)`.
- `pixel_maps_manifest.parquet` : cartes compactées des décisions pixel directes.
- `spatial_components.parquet` et `spatial_component_metrics.parquet` : diagnostics spatiaux raw / verrouillés.
- `validation_guardrails.parquet` : acceptabilité stricte 04C par exécution et scope, sans score composite.
- `validation_technical_events.parquet` : erreurs techniques reliées aux `fit_id` / `projection_id`.
- `validation_protocol.json` : lineage complète, hashes et contrat d'évaluation 04C.

Le notebook 05 devra consommer ces sorties **sans réintroduire les anciens identifiants 04C**.